### Import packages

In [10]:
from pathlib import Path
import json
import textdescriptives as td
from tqdm import tqdm
import pandas as pd
import numpy as np
import math

### Define paths and data

In [11]:
# Define project and data paths
PROJECT_ROOT = Path("/Users/tildeidunsloth/Desktop/Thesis")
DATA_DIR = PROJECT_ROOT / "data/cleaned"

science_fiction_data_path = DATA_DIR / "sci_fi_stories_cleaned.jsonl"
romance_data_path = DATA_DIR / "romance_stories_cleaned.jsonl"
literary_fiction_data_path = DATA_DIR / "lit_fiction_stories_cleaned.jsonl"

In [12]:
story_type = "lit_fiction"  # Change this to "sci_fi" or "romance" as needed
DATA_DIR_OUTPUT = PROJECT_ROOT / "data/processed"
OUTPUT_FILE = DATA_DIR_OUTPUT / f"{story_type}_stories_text_descriptors.jsonl"

In [13]:
# Read JSONL file
with open(literary_fiction_data_path, 'r') as f:
    literary_fiction_stories = [json.loads(line) for line in f]

In [14]:
def count_words(text):
    return len(text.split())

In [15]:
for item in tqdm(literary_fiction_stories, desc="Processing stories"):

    item["word_count"] = count_words(item["story"])

    item.update()

Processing stories: 100%|██████████| 5800/5800 [00:00<00:00, 17600.14it/s]


In [16]:
# count total words across all stories
total_words = sum(item["word_count"] for item in literary_fiction_stories)
print(f"Total words across all stories: {total_words}")

Total words across all stories: 10117753


### Get text descriptive measures

In [17]:
# possible metrics
td.get_valid_metrics()

{'all',
 'coherence',
 'dependency_distance',
 'descriptive_stats',
 'information_theory',
 'pos_proportions',
 'quality',
 'readability'}

In [18]:
total = len(literary_fiction_stories)
last_printed_percent = -1

with tqdm(total=100, desc="Processing stories") as pbar:
    for i, item in enumerate(literary_fiction_stories, start=1):
        metrics = td.extract_metrics(
            text=item["story"],
            metrics=['all'],
            spacy_model="en_core_web_lg"
        )

        cleaned_metrics = {}
        for key, value in metrics.items():
            if key == "text":
                continue
            if isinstance(value, pd.Series):
                cleaned_metrics[key] = value.iloc[0]
            else:
                cleaned_metrics[key] = value

        item.update(cleaned_metrics)

        # Compute integer percentage
        current_percent = math.floor((i / total) * 100)

        # Only update tqdm if percentage increased
        if current_percent > last_printed_percent:
            pbar.update(current_percent - last_printed_percent)
            last_printed_percent = current_percent

Processing stories:  26%|██▌       | 26/100 [1:21:53<4:02:06, 196.31s/it]/Users/tildeidunsloth/Desktop/Thesis/Thesis_project/lib/python3.12/site-packages/textdescriptives/components/coherence.py:44: UserWarning: [W008] Evaluating Span.similarity based on empty vectors.
  similarities.append(sent.similarity(sents[i + order]))
Processing stories: 101it [5:29:06, 195.51s/it]                          


In [19]:
# Pick the metrics you care about
keys_to_keep = [
    "story",
    "genre",
    "story_hint",
    "agent",
    "agent_type",
    "event",
    "event_valence",
    "context",
    "context_valence",
    "model",
    "date_of_generation",
    "output_tokens",
    "first_order_coherence", # how well each sentence connects to the previous one. Closer to 1 = smoother flow.
    "second_order_coherence", # coherence across larger spans (e.g., paragraphs). Higher values = more globally coherent text.
    "lix", # readability - Higher is more difficult. 20–30 is easy; 40–50 is difficult.
    "dependency_distance_mean", # coherence/structure - Average “distance” between syntactically linked words. Higher = more complex sentences.
    "mean_word_length", # mean length of words; higher = more complex vocabulary.
    "proportion_unique_tokens", # Lexical diversity; higher = more varied vocabulary.
    "pos_prop_ADJ", # % of adjectives
    "pos_prop_NOUN", # % of nouns
    "pos_prop_VERB", # % of verbs
    "pos_prop_PRON", # % of pronouns
    "duplicate_ngram_chr_fraction_5", # repetitiveness - High value → story repeats the same phrases or chunks of text often
    "passed_quality_check", # quality
    "oov_ratio", # fraction of out-of-vocabulary words
    "word_count" # number of words in the story
]

In [20]:
with open(OUTPUT_FILE, "w") as f:
    for item in tqdm(literary_fiction_stories, desc="Saving JSONL"):
        # Keep only selected keys
        filtered_item = {k: item[k] for k in keys_to_keep if k in item}

        # Convert numpy types and round floats
        for k, v in filtered_item.items():
            if isinstance(v, (np.bool_, np.bool8)):
                filtered_item[k] = bool(v)
            elif isinstance(v, (np.integer, np.int32, np.int64)):
                filtered_item[k] = int(v)
            elif isinstance(v, (np.floating, np.float32, np.float64)):
                filtered_item[k] = round(float(v), 2)  # round floats to 2 decimals

        # Write as one JSON object per line
        f.write(json.dumps(filtered_item) + "\n")

Saving JSONL:   0%|          | 0/5800 [00:00<?, ?it/s]/var/folders/wv/k1c_2q2x52q536wp2_p2kdpm0000gn/T/ipykernel_14079/527634504.py:8: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if isinstance(v, (np.bool_, np.bool8)):
Saving JSONL: 100%|██████████| 5800/5800 [00:01<00:00, 3755.48it/s]
